**1. Set up**

This notebook is where we evaluate the models. We are loading the models, running them on the test set, measuring how well they perform, and finding the tweets where they disagree. Those disagreement cases get saved for when we do the XAI analysis. We start by mounting Drive, importing everything we need, and confirming all 4 model paths are accessible.

In [ ]:

# mount Google Drive and import all libraries needed

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


PROJECT_PATH = '/content/drive/Shareddrives/Cos760'

# label mappings
LBL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LBL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LANGUAGES = ['hausa', 'kinyarwanda']

# model paths
MODEL_PATHS = {
    'xlmr': {
        lang: os.path.join(PROJECT_PATH, f'models/xlmr/{lang}/final')
        for lang in LANGUAGES
    },
    'afriberta': {
        lang: os.path.join(PROJECT_PATH, f'models/afriberta/{lang}/final')
        for lang in LANGUAGES
    }
}

print("Setup complete. Model paths:")
for model_name, langs in MODEL_PATHS.items():
    for lang, path in langs.items():
        exists = os.path.exists(path)
        print(f"  [{('OK' if exists else 'MISSING')}] {model_name}/{lang}: {path}")

Mounted at /content/drive
Using device: cuda
Setup complete. Model paths:
  [OK] xlmr/hausa: /content/drive/Shareddrives/Cos760/models/xlmr/hausa/final
  [OK] xlmr/kinyarwanda: /content/drive/Shareddrives/Cos760/models/xlmr/kinyarwanda/final
  [OK] afriberta/hausa: /content/drive/Shareddrives/Cos760/models/afriberta/hausa/final
  [OK] afriberta/kinyarwanda: /content/drive/Shareddrives/Cos760/models/afriberta/kinyarwanda/final


**2. Load test data**

In this cell we load the test splits for both Hausa and Kinyarwanda from the cleaned CSVs produced in notebook 01. We then print the label distribution for each language so we have a clear picture of class balance going into evaluation .

In [ ]:

#loading test data
test_dfs = {}

for lang in LANGUAGES:
    path = os.path.join(PROJECT_PATH, 'data/processed', f'{lang}_test_cleaned.csv')
    df = pd.read_csv(path)
    test_dfs[lang] = df




HAUSA test set: 5303 samples
  Label distribution:
label
neutral     1789
negative    1759
positive    1755

KINYARWANDA test set: 1026 samples
  Label distribution:
label
neutral     393
negative    355
positive    278


**3. Dataset class and inference function**

Here we define the two building blocks needed to run inference. The SentimentDataset class (from notebook 02) wraps our dataframe into a format PyTorch can read batch by batch. The run_inference() function takes a saved model path and a test dataframe, loads the model, and returns a predicted label for every individual tweet. Capturing per-tweet predictions is what allows us to do the disagreement analysis later.